In [ ]:
import pandas as pd
from pathlib import Path
import geopandas as gpd
import geobr
from shapely.geometry import Point
import folium

# Usamos o Pathlib aqui para gerenciar os caminhos das pastas.
pathDiario = Path("focos_diarios")      # Pasta onde salvamos o histórico diário do INPE
pathMinuto = Path("focos_por_minuto")    # Pasta com as atualizações de tempo real (10 min)

# Carregamos a base diária.
df_diario = pd.read_csv(pathDiario / "focos_diario_br_20260701.csv")

# Faxina nos dados: jogamos fora colunas redundantes para deixar o DataFrame leve.
# Como a nossa base já é exclusiva do Brasil, as colunas 'pais' e 'pais_id' só
# ocupam memória à toa. Também dropamos o 'satelite' daqui.
df_diario = df_diario.drop("satelite", axis=1)
df_diario = df_diario.drop("pais_id", axis=1)
df_diario = df_diario.drop("pais", axis=1)

# Criando um filtro para isolar os focos onde a precipitação NÃO é zero.
resultado = df_diario[df_diario['precipitacao'] != 0]

# Exibimos os primeiros 10 casos de focos com chuva para validar a filtragem
display(resultado.head(10))

# Mostramos os dataframes limpos para garantir que as colunas foram dropadas certo
display(df_diario)

,id,lat,lon,data_hora_gmt,municipio,estado,municipio_id,estado_id,numero_dias_sem_chuva,precipitacao,risco_fogo,bioma,frp
553,80265d52-5c96-3ff4-ac24-9a798f2b9bb9,-25.289301,-53.869202,2026-07-01 01:37:00,CÉU AZUL,PARANÁ,4105300,41,4,2.19,0.01,Mata Atlântica,NaN
981,c7511eee-a499-3680-8401-9dbdf24413e3,-2.850740,-40.190950,2026-07-01 04:00:00,ACARAÚ,CEARÁ,2300200,23,-999,0.10,0.54,Caatinga,0.9
1074,0a6bf52c-1d61-305d-b444-0a343289750a,-7.217940,-34.956300,2026-07-01 04:02:00,SANTA RITA,PARAÍBA,2513703,25,-999,0.30,0.00,Mata Atlântica,0.6
1250,ed9f8b9d-82ce-35cd-ae2a-de2b550d4bbc,-15.439270,-41.372000,2026-07-01 04:04:00,CÂNDIDO SALES,BAHIA,2906709,29,18,0.53,0.92,Mata Atlântica,1.0
4006,441eda49-9f60-3ba7-9aca-d648383ecd6b,-9.899110,-35.939870,2026-07-01 15:52:00,ROTEIRO,ALAGOAS,2707800,27,-999,0.60,-999.00,Mata Atlântica,2.6
4068,5dafef1b-ae22-3b5e-b02d-69990579ff6f,-8.771240,-39.667230,2026-07-01 15:52:00,CURAÇÁ,BAHIA,2909901,29,49,0.30,1.00,Caatinga,10.0
4084,1795fce3-51d8-3e94-9211-467145ebcb3f,-9.492410,-36.840690,2026-07-01 15:52:00,CACIMBINHAS,ALAGOAS,2701209,27,3,0.50,0.98,Caatinga,2.3
4526,3a7791ea-93b7-3950-aef4-b14b22f9c242,-9.821870,-37.234210,2026-07-01 16:37:00,BELO MONTE,ALAGOAS,2700904,27,1,0.60,0.99,Caatinga,2.0
4557,63b1feae-3ec9-3e44-9b55-b4e378423494,-9.095280,-40.284060,2026-07-01 16:37:00,PETROLINA,PERNAMBUCO,2611101,26,53,0.10,-999.00,Caatinga,13.9
4630,5436c885-aae9-38ec-9f43-60ea4abf88f9,-17.271640,-39.553130,2026-07-01 16:37:00,ALCOBAÇA,BAHIA,2900801,29,-999,0.10,1.00,Mata Atlântica,3.2


,id,lat,lon,data_hora_gmt,municipio,estado,municipio_id,estado_id,numero_dias_sem_chuva,precipitacao,risco_fogo,bioma,frp
0,74acd331-e951-30bf-8654-17a928c63e4a,-8.59330,-48.00690,2026-07-01 00:00:00,ITAPIRATINS,TOCANTINS,1710904,17,13,0.0,1.00,Cerrado,55.9
1,0a49bb04-cf96-3a2c-89c9-f99e45bf4891,-12.77460,-44.51620,2026-07-01 00:00:00,BAIANÓPOLIS,BAHIA,2902500,29,39,0.0,1.00,Cerrado,54.7
2,e49fae1e-c661-3678-b6c7-9f247fab8a7d,-16.36770,-56.67200,2026-07-01 00:00:00,POCONÉ,MATO GROSSO,5106505,51,6,0.0,1.00,Pantanal,67.8
3,f4474d6e-4bbe-376e-b265-54a02d7e6255,-11.83920,-56.94010,2026-07-01 00:00:00,PORTO DOS GAÚCHOS,MATO GROSSO,5106802,51,8,0.0,0.47,Amazônia,78.3
4,f4bfe20b-640c-34eb-a679-9c4d402e0328,-9.60370,-56.92760,2026-07-01 00:00:00,PARANAÍTA,MATO GROSSO,5106299,51,6,0.0,0.58,Amazônia,56.6
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8420,70e2d33d-983b-3eb8-80ba-13c8900cdfb6,-21.45547,-51.36207,2026-07-01 23:58:00,JUNQUEIRÓPOLIS,SÃO PAULO,3526001,35,6,0.0,1.00,Mata Atlântica,38.6
8421,b4e68246-4e13-3f5a-af4b-9f3c85f02fd4,-19.17235,-47.35860,2026-07-01 23:58:00,PERDIZES,MINAS GERAIS,3149804,31,5,0.0,0.66,Cerrado,23.4
8422,cae6a53f-bdea-3632-8c9d-11b3c1451d39,-22.54232,-46.97007,2026-07-01 23:58:00,MOGI MIRIM,SÃO PAULO,3530805,35,5,0.0,1.00,Cerrado,6.9
8423,88393daf-e7a8-3b0a-a1ba-806d56b5fa98,-18.67254,-52.70389,2026-07-01 23:58:00,CHAPADÃO DO CÉU,GOIÁS,5205471,52,5,0.0,0.98,Cerrado,24.9


In [3]:
pathMinuto = Path("focos_por_minuto")

df = pd.read_csv(pathMinuto/"focos_10min_20260629_0000.csv")
df= df.drop("satelite", axis=1)

display(df)

,lat,lon,data
0,-15.9945,-61.2144,2026-06-28 23:40:00
1,-15.9948,-61.1946,2026-06-28 23:40:00
2,-16.0141,-61.1930,2026-06-28 23:40:00
3,-16.1305,-71.6064,2026-06-28 23:40:00
4,-16.8796,-60.3805,2026-06-28 23:40:00
5,-16.8798,-60.3605,2026-06-28 23:40:00
6,-16.8991,-60.3786,2026-06-28 23:40:00
7,-16.8994,-60.3586,2026-06-28 23:40:00
8,-16.9187,-60.3768,2026-06-28 23:40:00
9,-16.9189,-60.3568,2026-06-28 23:40:00


In [4]:
# Inicializamos o mapa calculando a média das coordenadas de latitude e longitude.
# Isso garante que o mapa abra centralizado exatamente na região onde os focos
# ativos estão concentrados, evitando telas vazias ou fora do território do Brasil.
mapa = folium.Map(
    location=[df["lat"].mean(), df["lon"].mean()],
    zoom_start=4
)

# Percorremos o DataFrame de curto prazo para desenhar os alertas mais recentes.
# Usamos 'CircleMarker' com raio pequeno (3).
# A cor azul destaca o que ocorre AGORA.
for _, linha in df.iterrows():
    folium.CircleMarker(
        location=[linha["lat"], linha["lon"]],
        radius=3,
        color = 'blue'
    ).add_to(mapa)

# Percorremos a base diária para plotar o acumulado do dia inteiro.
# A cor vermelha serve para destinguir os focos azulados de 10 min dos diarios.
for _, linha in df_diario.iterrows():
    folium.CircleMarker(
        location=[linha["lat"],linha["lon"]],
        radius=3,
        color = 'red'
    ).add_to(mapa)
# Invocamos o objeto do mapa para renderizar a camada interativa no notebook.
mapa

In [5]:
# Cria os pontos geométricos usando as colunas de longitude e latitude
geometria = [Point(xy) for xy in zip(df['lon'], df['lat'])]

# Converte o DataFrame para GeoDataFrame com o sistema de coordenadas EPSG:4326
gdf_queimadas = gpd.GeoDataFrame(df, geometry=geometria, crs="EPSG:4326")

# Carrega os arquivos locais dos mapas de municípios e biomas
mapa_municipios = gpd.read_file("mapa_municipios_br.json")
mapa_biomas = gpd.read_file("mapa_biomas_br.json")

# Alinha o sistema de coordenadas dos mapas com o das queimadas
mapa_municipios = mapa_municipios.to_crs(gdf_queimadas.crs)
mapa_biomas = mapa_biomas.to_crs(gdf_queimadas.crs)

# Faz o cruzamento para identificar o município de cada foco
queimadas_com_municipio = gpd.sjoin(gdf_queimadas, mapa_municipios, how="left", predicate="within")

# Remove a coluna de índice do primeiro cruzamento para evitar duplicidade
if 'index_right' in queimadas_com_municipio.columns:
    queimadas_com_municipio = queimadas_com_municipio.drop(columns=['index_right'])

# Faz o segundo cruzamento espacial para identificar o bioma de cada foco
resultado_final = gpd.sjoin(queimadas_com_municipio, mapa_biomas, how="left", predicate="within")

# Seleciona apenas as colunas necessárias para a análise
colunas_interessantes = ['lat', 'lon','name_muni', 'abbrev_state', 'name_biome']
df_enriquecido = resultado_final[colunas_interessantes]

# Remove as linhas com valores nulos
df_enriquecido.dropna(inplace= True)

# Padroniza o nome dos municípios para letras maiúsculas
df_enriquecido['name_muni'] = df_enriquecido['name_muni'].str.upper()

# Exibe o DataFrame final enriquecido
display(df_enriquecido)

,lat,lon,name_muni,abbrev_state,name_biome
13,-8.7740,-63.6446,CANDEIAS DO JAMARI,RO,Amazônia
14,-9.0795,-59.4292,COLNIZA,MT,Amazônia
15,-9.0796,-59.4098,COLNIZA,MT,Amazônia
16,-9.0798,-59.3904,COLNIZA,MT,Amazônia
17,-9.3140,-58.0309,APIACÁS,MT,Amazônia
18,-9.3327,-58.0298,APIACÁS,MT,Amazônia
19,-9.3329,-58.0102,APIACÁS,MT,Amazônia
20,-10.3363,-50.1905,PIUM,TO,Cerrado
21,-10.3365,-50.1691,PIUM,TO,Cerrado
22,-11.2148,-53.8330,MARCELÂNDIA,MT,Amazônia


In [6]:

geometria = [Point(xy) for xy in zip(df_diario['lon'], df_diario['lat'])]

gdf_queimadas = gpd.GeoDataFrame(df_diario, geometry=geometria, crs="EPSG:4326")

queimadas_com_municipio = gpd.sjoin(gdf_queimadas, mapa_municipios, how="left", predicate="within")

if 'index_right' in queimadas_com_municipio.columns:
    queimadas_com_municipio = queimadas_com_municipio.drop(columns=['index_right'])

resultado_final = gpd.sjoin(queimadas_com_municipio, mapa_biomas, how="left", predicate="within")

colunas_interessantes = ['lat', 'lon','name_muni', 'abbrev_state', 'name_biome']
df_diario_enriquecido = resultado_final[colunas_interessantes]
df_diario_enriquecido.dropna(inplace= True)

display(df_diario_enriquecido)


,lat,lon,name_muni,abbrev_state,name_biome
0,-8.59330,-48.00690,Itapiratins,TO,Cerrado
1,-12.77460,-44.51620,Baianópolis,BA,Cerrado
2,-16.36770,-56.67200,Poconé,MT,Cerrado
3,-11.83920,-56.94010,Porto dos Gaúchos,MT,Amazônia
4,-9.60370,-56.92760,Paranaíta,MT,Amazônia
...,...,...,...,...,...
8420,-21.45547,-51.36207,Junqueirópolis,SP,Mata Atlântica
8421,-19.17235,-47.35860,Perdizes,MG,Cerrado
8422,-22.54232,-46.97007,Mogi Mirim,SP,Mata Atlântica
8423,-18.67254,-52.70389,Chapadão do Céu,GO,Cerrado


In [7]:
import pandas as pd
from pygbif import occurrences as occ

import pandas as pd
from pygbif import occurrences

def buscar_especies_por_coordenadas(lat, lon, raio_graus):
    """
    Busca ocorrências de espécies no GBIF dentro de uma área delimitada.
    
    Parâmetros:
    lat (float): Latitude central
    lon (float): Longitude central
    raio_graus (float): Raio para criar o bounding box (padrão ~5km)
    """
    # Define os limites geográficos 
    min_lat = lat - raio_graus
    max_lat = lat + raio_graus
    min_lon = lon - raio_graus
    max_lon = lon + raio_graus
    
    # Realiza a busca na API do GBIF
    resposta = occurrences.search(
        decimalLatitude=f"{min_lat},{max_lat}",
        decimalLongitude=f"{min_lon},{max_lon}",
        hasCoordinate=True,
        limit=50
    )
    
    registros = resposta.get('results', [])
    
    if not registros:
        print("Nenhuma ocorrência encontrada nesta localização.")
        return None
        
    # Extrai as informações principais de cada ocorrência
    dados_formatados = []
    for reg in registros:
        dados_formatados.append({
            'Espécie': reg.get('species', 'Não identificada'),
            'Nome Científico': reg.get('scientificName', 'N/A'),
            'Latitude': reg.get('decimalLatitude'),
            'Longitude': reg.get('decimalLongitude'),
            'Data do Registro': reg.get('eventDate', 'N/A'),
            'Base do Registro': reg.get('basisOfRecord', 'N/A')
        })
        
    # Converte para DataFrame do Pandas para melhor visualização
    df = pd.DataFrame(dados_formatados)
    return df

def adaptador_para_apply(linha):
    # Extrai a latitude e longitude da linha atual do DataFrame
    # (Ajuste os nomes 'lat' e 'lon' se no seu df_diario_enriquecido eles forem diferentes)
    lat = linha['lat']
    lon = linha['lon']
    
    # Coloque um pequeno delay para evitar o bloqueio de requisições.
    import time
    time.sleep(0.1) 
    
    # Chama a função original
    df_especies = buscar_especies_por_coordenadas(lat, lon, raio_graus=0.02)
    
    # Se a função encontrou espécies e retornou o DataFrame:
    if df_especies is not None and not df_especies.empty:
        # Pega a coluna 'Espécie', remove as duplicadas e junta tudo em um texto só
        especies_unicas = df_especies['Espécie'].dropna().unique()
        return ", ".join(especies_unicas)
    else:
        return "Nenhuma espécie registrada"

print("Processando linhas e consultando o GBIF...")

# Criando uma amostra de teste com poucas linhas.
df_teste = df_diario_enriquecido.head(10).copy()

# Agora usamos o adaptador que sabe ler a linha e devolver um texto simples
df_teste['especies_vizinhas'] = df_teste.apply(adaptador_para_apply, axis=1)

# Exibe o resultado final com a nova coluna preenchida de verdade.
display(df_teste)


Processando linhas e consultando o GBIF...
Nenhuma ocorrência encontrada nesta localização.
Nenhuma ocorrência encontrada nesta localização.
Nenhuma ocorrência encontrada nesta localização.
Nenhuma ocorrência encontrada nesta localização.
Nenhuma ocorrência encontrada nesta localização.
Nenhuma ocorrência encontrada nesta localização.


,lat,lon,name_muni,abbrev_state,name_biome,especies_vizinhas
0,-8.5933,-48.0069,Itapiratins,TO,Cerrado,Nenhuma espécie registrada
1,-12.7746,-44.5162,Baianópolis,BA,Cerrado,Nenhuma espécie registrada
2,-16.3677,-56.6720,Poconé,MT,Cerrado,"Palusophis bifossatus, Anodorhynchus hyacinthi..."
3,-11.8392,-56.9401,Porto dos Gaúchos,MT,Amazônia,Nenhuma espécie registrada
4,-9.6037,-56.9276,Paranaíta,MT,Amazônia,Nenhuma espécie registrada
5,-10.4387,-49.6005,Pium,TO,Cerrado,"Buteo brachyurus, Cyanocorax cyanopogon, Caria..."
6,-12.7557,-44.4959,Baianópolis,BA,Cerrado,Nenhuma espécie registrada
7,-5.8058,-43.7975,Buriti Bravo,MA,Cerrado,"Euphonia violacea, Notharchus tectus, Xiphocol..."
8,-10.4390,-49.5789,Pium,TO,Cerrado,Davilla elliptica
9,-12.4285,-58.3850,Brasnorte,MT,Amazônia,Nenhuma espécie registrada


In [8]:
vegatacao = geobr.read_


AttributeError: module 'geobr' has no attribute 'read_'